In [14]:
'''
import requests
import base64
import re
# 应用名称：全功能识别接口
from config import API_KEY, SECRET_KEY   # 从 config 导入
    
def get_access_token():
    """
    使用 AK，SK 生成鉴权签名（Access Token）
    :return: access_token，或是None(如果错误)
    """
    url = "https://aip.baidubce.com/oauth/2.0/token"
    params = {"grant_type": "client_credentials", "client_id": API_KEY, "client_secret": SECRET_KEY}
    return str(requests.post(url, params=params).json().get("access_token"))

def extract_student_info(result_json):
    words_list = result_json.get('words_result', [])
    
    student_id = None
    id_location = None
    
    # 找学号
    for item in words_list:
        word = item['words']
        if re.match(r'^\d{12}$', word):
            student_id = word
            id_location = item['location']
            break
    
    if not student_id:
        return None, None
    
    # 找离学号最近的姓名
    def distance(loc):
        return ((loc['top'] - id_location['top']) ** 2 + (loc['left'] - id_location['left']) ** 2) ** 0.5
    
    best_name = None
    min_dist = float('inf')
    
    for item in words_list:
        word = item['words']
        if re.match(r'^[\u4e00-\u9fa5]{2,4}$', word):
            d = distance(item['location'])
            if d < min_dist:
                min_dist = d
                best_name = word
    
    return student_id, best_name


def main():  
    url = "https://aip.baidubce.com/rest/2.0/ocr/v1/handwriting?access_token=" + get_access_token()
    # 读取本地图片
    with open("images_example/5.jpg", "rb") as f:
        img_base64 = base64.b64encode(f.read()).decode()
    # 构造 payload
    payload = {
        'image': img_base64,
        'recognize_granularity': 'big'
    }
    #测试图样payload='url=https%3A%2F%2Fbaidu-ai.bj.bcebos.com%2Focr%2Fhandwriting.jpeg&eng_granularity=letter&detect_direction=false&probability=false&detect_alteration=false'
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Accept': 'application/json'
    }
    response = requests.request("POST", url, headers=headers, data=payload)
    response.encoding = "utf-8"
    result = response.json()
    print(result)

    # 提取学号和姓名
    student_id, student_name = extract_student_info(result)
    
    print(f"学号：{student_id}")
    print(f"姓名：{student_name}")

if __name__ == '__main__':
    main()
'''

{'words_result': [{'location': {'top': 186, 'left': 2083, 'width': 130, 'height': 57}, 'words': 'Date.'}, {'location': {'top': 196, 'left': 2376, 'width': 148, 'height': 65}, 'words': 'Page.'}, {'location': {'top': 250, 'left': 271, 'width': 1093, 'height': 166}, 'words': '302025033134倩措姆.'}, {'location': {'top': 524, 'left': 166, 'width': 677, 'height': 152}, 'words': '1.import math'}, {'location': {'top': 665, 'left': 262, 'width': 1497, 'height': 157}, 'words': 'afloatlinput(“请输入边a的长度:”))'}, {'location': {'top': 793, 'left': 276, 'width': 1531, 'height': 162}, 'words': 'b=float(input(“请输入边b的长度:”))'}, {'location': {'top': 933, 'left': 284, 'width': 1605, 'height': 147}, 'words': 'C=floatlinput(“请输入边c的长度:”))'}, {'location': {'top': 1061, 'left': 287, 'width': 1119, 'height': 128}, 'words': 'c_rad=math、radians(c)'}, {'location': {'top': 1188, 'left': 289, 'width': 1326, 'height': 131}, 'words': 'S=0.5a米bmath.sin(C-rad)'}, {'location': {'top': 1318, 'left': 313, 'width': 1231, 'height':

In [26]:
import requests
import base64
import re
from config import API_KEY, SECRET_KEY
import time

def get_access_token():
    url = "https://aip.baidubce.com/oauth/2.0/token"
    params = {"grant_type": "client_credentials", "client_id": API_KEY, "client_secret": SECRET_KEY}
    return str(requests.post(url, params=params).json().get("access_token"))

def extract_student_info(result_json):
    words_list = result_json.get('words_result', [])
    
    student_id = None
    student_name = None
    id_location = None
    
    # 第一遍：找学号和姓名
    for item in words_list:
        text = item['words']
        
        # 情况1：学号和姓名在同一行
        match = re.search(r'(\d{12,15})([\u4e00-\u9fa5]{2,4})', text)
        if match:
            student_id = match.group(1)
            student_name = match.group(2)
            id_location = item['location']
            break
        
        # 情况2：单独学号
        if not student_id and re.match(r'^\d{12,15}$', text):
            student_id = text
            id_location = item['location']
        
        '''
        # 情况3：单独姓名（纯中文）
        if not student_name and re.match(r'^[\u4e00-\u9fa5]{2,4}$', text):
            student_name = text
        '''
    
    # 按距离找名字
    if student_id and not student_name and id_location:
        def distance(loc):
            return ((loc['top'] - id_location['top']) ** 2 + (loc['left'] - id_location['left']) ** 2) ** 0.5
        
        best_name = None
        min_dist = float('inf')
        
        for item in words_list:
            word = item['words']
            if re.search(r'[\u4e00-\u9fa5]{2,4}', word):  # 包含中文就行
                # 提取纯中文部分
                name = re.sub(r'[^\u4e00-\u9fa5]', '', word)
                if len(name) >= 2:
                    d = distance(item['location'])
                    if d < min_dist:
                        min_dist = d
                        best_name = name
        
        if best_name:
            student_name = best_name
    
    return student_id, student_name

def recognize_handwriting(image_path):
    """识别单张图片"""
    token = get_access_token()
    url = f"https://aip.baidubce.com/rest/2.0/ocr/v1/handwriting?access_token={token}"
    
    with open(image_path, "rb") as f:
        img_base64 = base64.b64encode(f.read()).decode()
    
    payload = {
        'image': img_base64,
        'recognize_granularity': 'big',
    }
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    
    start = time.time()
    response = requests.post(url, headers=headers, data=payload)
    student_id, student_name = extract_student_info(response.json())
    end = time.time()
    elapsed = end - start
    return student_id, student_name, elapsed

def main():  

    image_path = "images_example/5_cropped.jpg"
    #image_path = "images_example/5.jpg"
    
    url = "https://aip.baidubce.com/rest/2.0/ocr/v1/handwriting?access_token=" + get_access_token()
    
    student_id, student_name, time = recognize_handwriting(image_path)
    
    print(f"学号：{student_id}")
    print(f"姓名：{student_name}")
    print(f"耗时：{time}")
    
if __name__ == '__main__':
    main()

学号：302025033134
姓名：倩措姆
耗时：4.704212427139282


### 测试时间

In [20]:
'''
import requests
import base64
import time
from config import API_KEY, SECRET_KEY

def get_access_token():
    url = "https://aip.baidubce.com/oauth/2.0/token"
    params = {
        "grant_type": "client_credentials",
        "client_id": API_KEY,
        "client_secret": SECRET_KEY
    }
    return requests.post(url, params=params).json().get("access_token")

def recognize_handwriting(image_path):
    """识别单张图片，返回识别结果和耗时"""
    token = get_access_token()
    url = f"https://aip.baidubce.com/rest/2.0/ocr/v1/handwriting?access_token={token}"
    
    with open(image_path, "rb") as f:
        img_base64 = base64.b64encode(f.read()).decode()
    
    payload = {
        'image': img_base64,
        'recognize_granularity': 'big',
    }
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    
    start = time.time()
    response = requests.post(url, headers=headers, data=payload)
    end = time.time()
    
    elapsed = end - start
    return response.json()

def main():
    # 图片路径（改成你自己的路径）
    full_image_path = "images_example/5.jpg"          # 未裁剪原图
    cropped_image_path = "images_example/5_cropped.jpg"  # 裁剪后的图
    
    print("开始识别未裁剪图片...")
    result_full, time_full = recognize_handwriting(full_image_path)
    print(f"未裁剪图片耗时：{time_full:.3f} 秒")
    
    print("\n开始识别裁剪图片...")
    result_cropped, time_cropped = recognize_handwriting(cropped_image_path)
    print(f"裁剪图片耗时：{time_cropped:.3f} 秒")
    
    print(f"\n速度提升：{time_full / time_cropped:.1f} 倍")

if __name__ == '__main__':
    main()

开始识别未裁剪图片...
未裁剪图片耗时：3.358 秒

开始识别裁剪图片...
裁剪图片耗时：0.305 秒

速度提升：11.0 倍
